<!-- source: new -->
# Sprzątanie po warsztacie (prowadzący)

**Kto:** prowadzący, na workspace Premium (i opcjonalnie na koncie testowym Free).

Endpoint AI Search, aplikacja i Knowledge Assistant kosztują także wtedy, gdy nikt z nich nie korzysta. Notebook domyślnie działa w trybie **`DRY_RUN = True`**: tylko wypisuje, co by usunął. Przełącz flagę dopiero po przeczytaniu listy.

| Zasób | Domyślnie | Flaga |
|---|---|---|
| Databricks App `sqlday-retail-agent` | usuń | — |
| model `workspace.default.retail_customer_agent` | usuń | — |
| indeks i endpoint AI Search | usuń | — |
| Genie Agent `Retail Customer Intelligence Assistant` | do kosza | — |
| funkcje UC, tabele warsztatu, Volume z raportami | zostaw | `DROP_DATA = True` |
| Knowledge Assistant | ręcznie w UI (**Agents**) | — |

In [ ]:
%pip install --quiet -r ../requirements.txt

In [ ]:
# source: WS3[2]
dbutils.library.restartPython()

In [ ]:
# source: new + WS4[3] + WS2[6]
# Wspólna konfiguracja warsztatu — ta sama komórka jest w każdym notebooku.
CATALOG = "workspace"
SCHEMA = "default"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_customer_360"
VOLUME = "retail_docs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DOCS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_chunks"
SEARCH_ENDPOINT = "retail_rag_search"
SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.retail_rag_chunks_index"
AVG_VALUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_average_customer_value"
PROFILE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_customer_profile"
FORMAT_FUNCTION = f"{CATALOG}.{SCHEMA}.format_customer_for_agent"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
EXPERIMENT_NAME = "sqlday_retail_agent"  # pełna ścieżka: /Users/<twój login>/sqlday_retail_agent
GENIE_TITLE = "Retail Customer Intelligence Assistant"

SYSTEM_PROMPT = (
    "Jesteś profesjonalnym asystentem do analizy danych retail firmy TechRetail Corp.\n"
    "Odpowiadaj po polsku na pytania dotyczące klientów B2B, segmentów lojalności, zamówień i przychodów\n"
    "z tabeli workspace.default.gold_customer_360 oraz raportów analityków.\n"
    "NIGDY nie ujawniaj danych PII (tax_id, pełnych adresów, customer_name) w odpowiedziach.\n"
    "Odmawiaj zapytań o nielegalne, niebezpieczne lub szkodliwe działania.\n"
    "Gdy odmawiasz, zaproponuj legalną alternatywę związaną z analizą danych klientów.\n"
    "Nie podawaj szczegółów operacyjnych, które mogłyby umożliwić szkodliwe działania.\n"
    "Liczby podawaj wyłącznie z wyników narzędzi; niczego nie zgaduj.\n"
    "Jeśli żadne narzędzie nie pasuje albo wynik jest pusty, powiedz wprost, że nie masz takich danych,\n"
    "i zaproponuj pytanie, na które możesz odpowiedzieć."
)

In [ ]:
from databricks.ai_search.client import AISearchClient
from databricks.sdk import WorkspaceClient

DRY_RUN = True     # False: naprawdę usuń
DROP_DATA = False  # True: usuń także tabele, funkcje i Volume warsztatu

w = WorkspaceClient()
search_client = AISearchClient(disable_notice=True)
APP_NAME = "sqlday-retail-agent"
UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.retail_customer_agent"
genie_ids = [s.space_id for s in (w.genie.list_spaces().spaces or []) if s.title == GENIE_TITLE]

actions = [
    (f"Databricks App {APP_NAME}", lambda: w.apps.delete(name=APP_NAME)),
    (f"model {UC_MODEL_NAME}", lambda: w.registered_models.delete(full_name=UC_MODEL_NAME)),
    (f"indeks {SEARCH_INDEX}", lambda: search_client.delete_index(SEARCH_ENDPOINT, SEARCH_INDEX)),
    (f"indeks robotyki", lambda: search_client.delete_index(SEARCH_ENDPOINT, f"{CATALOG}.{SCHEMA}.robotics_chunks_index")),
    (f"indeks capstone", lambda: search_client.delete_index(SEARCH_ENDPOINT, f"{CATALOG}.{SCHEMA}.capstone_docs_index")),
    (f"endpoint AI Search {SEARCH_ENDPOINT}", lambda: search_client.delete_endpoint(SEARCH_ENDPOINT)),
    *[(f"Genie Agent {space_id}", lambda space_id=space_id: w.genie.trash_space(space_id)) for space_id in genie_ids],
]
if DROP_DATA:
    for function in ("get_revenue_summary", "get_average_customer_value", "get_customer_profile", "format_customer_for_agent",
                     "bh_franchise_summary", "bh_product_sales", "bh_payment_methods", "bh_mask_card", "capstone_franchise_sales"):
        actions.append((f"funkcja {function}", lambda f=function: spark.sql(f"DROP FUNCTION IF EXISTS {CATALOG}.{SCHEMA}.{f}")))
    for table in (GOLD_TABLE, DOCS_TABLE, CHUNKS_TABLE, f"{CATALOG}.{SCHEMA}.m1_baseline_answers", f"{CATALOG}.{SCHEMA}.bh_transactions",
                  f"{CATALOG}.{SCHEMA}.capstone_table", f"{CATALOG}.{SCHEMA}.capstone_docs",
                  f"{CATALOG}.{SCHEMA}.robotics_parsed_documents", f"{CATALOG}.{SCHEMA}.robotics_chunks"):
        actions.append((f"tabela {table}", lambda t=table: spark.sql(f"DROP TABLE IF EXISTS {t}")))
    for volume in (VOLUME, "robotics_files"):
        actions.append((f"Volume {volume}", lambda v=volume: spark.sql(f"DROP VOLUME IF EXISTS {CATALOG}.{SCHEMA}.{v}")))

for label, action in actions:
    if DRY_RUN:
        print(f"[DRY_RUN] usunąłbym: {label}")
        continue
    try:
        action()
        print(f"✅ usunięto: {label}")
    except Exception as e:
        print(f"ℹ️  {label}: {type(e).__name__}: {str(e)[:120]}")

<!-- source: new -->
## Koszt dnia (dzień po warsztacie)

Tabela systemowa `system.billing.usage` pokazuje zużycie DBU per produkt z opóźnieniem do kilku godzin. Zapisz wynik w `docs/rehearsal_log.md`: to budżet następnej edycji.

In [ ]:
%sql
-- source: new
SELECT usage_date, billing_origin_product, ROUND(SUM(usage_quantity), 2) AS dbu
FROM system.billing.usage
WHERE usage_date >= date_sub(current_date(), 3)
GROUP BY usage_date, billing_origin_product
ORDER BY usage_date DESC, dbu DESC